# Подключение к Spark

Тетрадка для Jupyter на платформе (`/home/jovyan`). Локальный Windows-проект озеро не видит.

Конспект рабочей тетрадки коллеги: там Spark уже поднимается. Ниже только то, что нужно для входа. Загрузка в Oracle и бизнес-логика той тетрадки сюда не перенесены.

## Как у коллеги устроено подключение

Два входа, один keytab.

1. Логин и пароль в файле не лежат. Ячейка спрашивает их так же, как остальные тетрадки: `input` и `getpass`. Keytab тот же, что у Impala в этих тетрадках: `/home/jovyan/test_requests/tech.keytab`, с `use_credentials` и `update_keytab`.
2. Impala открывается через `rail_connectors.connection.connect(to='IMPALA')`. База: `sandbox_ai`, очередь `ai`. После `imp._init_connection()` нужен успешный `SELECT 1`.
3. Spark **не** создаётся через `.master("yarn")` и **не** получает очередь в коде. Перед `SparkSession` каталог конфигурации платформы копируется в окружение:

```python
if "SPARK_K8S_CONF_DIR" in os.environ:
    os.environ["SPARK_CONF_DIR"] = os.environ["SPARK_K8S_CONF_DIR"]
```

Мастер, очередь и параметры Kerberos приходят из этого каталога. В билдере только два явных параметра: `spark.submit.deployMode = client` и `spark.sql.catalogImplementation = hive`. Второе нужно, чтобы `spark.table("sandbox_ai....")` и `spark.sql` видели те же таблицы, что Impala.
4. Старая сессия в том же ядре гасится через `spark.stop()` в `try/except`. Если переменной ещё нет, ловится `NameError`.
5. Чтение озера после этого: `spark.table("схема.таблица")` или `spark.sql(...)`. Запись в другую базу через JDBC в этой тетрадке не нужна.

## Почему прошлый ручной запуск падал

`SparkSession.builder.master("yarn")` без билета Kerberos доходит до HDFS и падает с `Failed to find any Kerberos tgt`. JVM умирает, Python показывает `JAVA_GATEWAY_EXITED`. В тетрадке коллеги билет появляется на шаге Impala (keytab), а мастер Spark берётся из `SPARK_CONF_DIR`, а не прописывается вручную.

## Порядок ячеек

1. Импорты. Логин и пароль спросят в следующей ячейке.
2. Impala и `SELECT 1`.
3. Сессия Spark.
4. Проверка: версия, `select 1`, список баз.

Пароль в тетрадку не вставлять: его запрашивает `getpass` и на экран не печатает.

## 1. Импорты

In [ ]:
import getpass
import logging
import os
import subprocess

from pyspark.sql import SparkSession
from rail_connectors.connection import connect

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("spark_connect")

keytab_path = "/home/jovyan/test_requests/tech.keytab"

print("keytab_exists", os.path.isfile(keytab_path))
print("SPARK_K8S_CONF_DIR", os.environ.get("SPARK_K8S_CONF_DIR"))
print("SPARK_CONF_DIR", os.environ.get("SPARK_CONF_DIR"))

## 2. Impala

При запуске ячейка спросит логин и пароль. Пароль на экран не выводится. Файла keytab может ещё не быть: `update_keytab` создаёт `/home/jovyan/test_requests/tech.keytab` из введённых логина и пароля, как в остальных тетрадках.

In [ ]:
os.makedirs(os.path.dirname(keytab_path), exist_ok=True)

LAKE_USER = input("Логин Impala: ").strip()
LAKE_PASSWORD = getpass.getpass("Пароль Impala: ")
if not LAKE_USER or not LAKE_PASSWORD:
    raise RuntimeError("Логин и пароль нужны оба")

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": keytab_path,
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": LAKE_USER, "password": LAKE_PASSWORD},
)
imp._init_connection()

with imp:
    imp.execute("SELECT 1")

logger.info("Impala: SELECT 1 прошёл")
print(subprocess.getoutput("klist"))

## 3. Сессия Spark

Если `SPARK_K8S_CONF_DIR` пустой, сессия пойдёт с системным `SPARK_CONF_DIR`. У коллеги рабочий путь — подменить его каталогом платформы. Мастер и очередь в этой ячейке не задаются.

In [ ]:
logger.info("Инициализация Spark сессии")

if "SPARK_K8S_CONF_DIR" in os.environ:
    os.environ["SPARK_CONF_DIR"] = os.environ["SPARK_K8S_CONF_DIR"]
    logger.info("SPARK_CONF_DIR = %s", os.environ["SPARK_CONF_DIR"])
else:
    logger.warning("SPARK_K8S_CONF_DIR не найден. SPARK_CONF_DIR не переопределён.")

try:
    spark.stop()
    logger.info("Текущая Spark сессия остановлена")
except NameError:
    logger.info("Spark сессия ещё не создана")
except Exception as e:
    logger.info("Нет живой сессии Spark: %s", e)

spark = (
    SparkSession.builder
    .appName("acq-dashboard-mvp")
    .config("spark.submit.deployMode", "client")
    .config("spark.sql.catalogImplementation", "hive")
    .getOrCreate()
)

logger.info("Spark сессия инициализирована")
print("version", spark.version)
print("ui", spark.sparkContext.uiWebUrl)

## 4. Проверка, что сессия считает

`select 1` не трогает таблицы. `show databases` только проверяет, что каталог Hive открылся. Полную витрину отсюда не запускать.

In [ ]:
ok = spark.sql("select 1 as ok")
ok.persist()
ok.show()

spark.sql("show databases").show(30, truncate=False)